<a href="https://colab.research.google.com/github/Sharddha-Sharddha/IITMLAssignments/blob/main/Legendary_Pok%C3%A9mon_decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

Problem Statement:

Pokémon is a group of adorable creatures peacefully colonizing a planet until humans come along and make them combat each other in order to get shiny badges and we can call them Pokémon masters.
In this universe, there exists a group of rare and often strong Pokémon, known as Legendary Pokémon. Unfortunately, there are no detailed criteria that define these Pokémon.
The only way to recognize a Legendary Pokémon is through information from official media, such as the game or anime.
This data set includes 721 Pokemon, including their number, name, first and second type, and basic stats: HP, Attack, Defense, Special Attack, Special Defense, and Speed. The legend of a pokemon cannot be suspected only by its Attack and Defense. It would be worth finding which variables can define the legend of a pokemon. The strategy is to analyze the data and perform a predictive task of classification to predict the legend of a pokemon using a decision tree algorithm.

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Data/Pokemon.csv')

In [ ]:
data

In [ ]:
data = data.drop(columns = ['#'], axis = 1)

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

# Task
Perform an exploratory data analysis (EDA) on the `data` DataFrame, including generating descriptive statistics for numerical columns,
checking for missing value placeholders (e.g., 'NaN') in object-type columns,
visualizing the distribution of the 'Legendary' variable.
Renaming `Type 1` and `Type 2` column  
The strategy is to analyze the data and perform a predictive task of classification to predict the legend of a pokemon using a decision tree algorithm.

**Reasoning**:
To generate descriptive statistics for the numerical columns, I will use the `.describe()` method on the `data` DataFrame as instructed.



In [ ]:
data.describe().T

In [ ]:
data.describe(include = 'O').T

In [ ]:
data= data.rename({'Type 1': 'Type1',
                   'Type 2': 'Type2'}, axis = 1)

In [ ]:
data.isnull().sum()

In [ ]:
from statistics import mode

In [ ]:
type_mode = data['Type2'].mode()[0]
type_mode

In [ ]:
#replacing null value with mode
data['Type2'] = data['Type2'].fillna(type_mode)

In [ ]:
data.isnull().sum()

In [ ]:
data.head(10)

In [ ]:
data.duplicated().sum()

No Duplicate Data Found

In [ ]:
numerical_features = [col for col in data.columns if data[col].dtype != 'object' and col != 'Legendary']
for col in numerical_features:
  plt.figure(figsize= (10,6))
  plt.boxplot(data[col])
  plt.xlabel(col)
  plt.show()

No need to remove Outliers since we are perfoming Decision Tress so Outlier will not impact much

## Visualize Distribution of 'Legendary'



In [ ]:
plt.figure(figsize= (10,6))
ax = sns.countplot(data = data, x='Legendary', hue='Legendary', legend = True, palette='viridis')
plt.xlabel('Legendary')
plt.title('Distribution of Legendary Pokémon')
plt.xticks(rotation = 45)
plt.show()

## Building the Machine Learning Model ♟

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score,roc_curve

In [ ]:
data.info()

In [ ]:
le = LabelEncoder()
for column in data.columns:
  if (data[column].dtype == 'object')& (column != 'Legendary'):
    data[column]= le.fit_transform(data[column])

In [ ]:
data.info()

In [ ]:
# spliting the data for training the model
x = data.drop(['Legendary'], axis = 1)
y = data['Legendary']

In [ ]:
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size= 0.20, random_state = 42, stratify = y)

In [ ]:
x_train.shape

In [ ]:
x_test.shape

In [ ]:
y_train.shape

In [ ]:
y_test.shape

#### **Standardization / Scaling of the data**

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
DT = DecisionTreeClassifier(random_state = 42)

DT.fit(x_train_scaled,y_train)

y_pred = DT.predict(x_test_scaled)
accuracy = accuracy_score(y_test,y_pred)

print(f'Decisio Tree Accuracy(Base model) : {accuracy}')

grid_params = {'criterion': ['gini','entropy'],
               'min_samples_leaf' : [1,3,5],
               'min_samples_split' : [2,3,6,8,9],
               'max_depth': [None,2,5,7],
               'max_features':[3,5,7,9]
              }

RS = RandomizedSearchCV(estimator = DT, param_distributions =  grid_params, cv = 5, n_iter = 70,  n_jobs = -1)

RS.fit(x_train_scaled, y_train)

print("Best params:", RS.best_params_)
print("Best CV score:", RS.best_score_)

# Optionally, evaluate the best estimator found by RandomizedSearchCV

tuned_model = RS.best_estimator_

y_pred_tuned = tuned_model.predict(x_test_scaled)

accuracy_tuned = accuracy_score(y_test,y_pred_tuned)

print(f'Decisio Tree Accuracy (Tuned model) : {accuracy_tuned}')


Let's compare the performance of the base Decision Tree model and the tuned Decision Tree model based on their accuracy scores:


* Base Decision Tree Accuracy: 0.95 (95%)
* Tuned Decision Tree Accuracy: 0.96875(96.87%)


In [ ]:
pd.DataFrame({'Actual' : y_test,
              'Predicted' : y_pred_tuned})

In [ ]:
cf = confusion_matrix(y_test,y_pred_tuned)
cf

Our Model Is preforming very well.
By using confusion matrix we conclude that from total 16o predictions model only predict 5 wrong prediction.

ROC Cuve

In [ ]:
# `roc_auc_score` requires numerical labels for y_test.
# label encoding on y_test
y_test_encoded = le.fit_transform(y_test)


In [ ]:
y_pred_proba = tuned_model.predict_proba(x_test_scaled)[:, 1]
y_pred_proba

In [ ]:
roc_auc_score(y_test_encoded,y_pred_proba)* 100

98.7% ROC AUC means our
 model correctly ranks positive vs negative cases 98.7% of the time.

 This suggests that your Decision Tree model is highly effective at identifying Legendary Pokémon.


# Plot ROC curve

In [ ]:
plt.figure(figsize = (10,6))
fpr, tpr, thresholds = roc_curve(y_test_encoded, y_pred_proba)
plt.plot(fpr, tpr, color = 'blue', label = (f'ROC curve : area = {roc_auc_score(y_test_encoded,y_pred_proba) : .2f}'))
plt.plot([0,1],[0,1], linestyle = '--', color = 'red', label = ' Random Classification' )
plt.xlabel('False Positve Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.grid(alpha = 0.5)
plt.show()

## Conclusion

This ROC curve shows our model is separating the two classes extremely well.

In summary, the plot visually confirms that your Decision Tree model (the blue curve) is performing significantly better than random guessing (the red dashed line) at classifying Legendary Pokémon.


# Quiz


1) How many pokemon are from the 5th generation?

In [ ]:
data.head()

In [ ]:
data[data['Generation'] == 5].value_counts().shape[0]

2) How many pokemon have the highest defense score?

In [ ]:
max_defence = data['Defense'].max()
data[data['Defense']== max_defence].value_counts().shape[0]

3) Which column of the following column is having negative relationship with the generation column?

In [ ]:
import seaborn as sns
plt.figure(figsize = (10,6))
sns.heatmap(data.corr(), annot = True)
plt.show()

speed

In [ ]:
classification_report(y_test_encoded,y_pred_tuned)

In [ ]:
dib = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Data/diabetes.csv')

In [ ]:
dib

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
oe = OrdinalEncoder()

In [ ]:
def bp_categories(bp):
  if bp<80:
    return 'Low'

  elif(80<=bp<=120):
    return 'Normal'

  else:
    return 'High'


dib['bp_cat'] = dib['BloodPressure'].apply(bp_categories)
dib

#  Entropy:  - Σ p * log2(p)

p = dib["bp_cat"].value_counts(normalize=True)   # probabilities
p

entropy= -(p* np.log2(p)).sum()
print("Entropy:", entropy)


In [ ]:
import numpy as np

for col in dib.columns:
  if col != 'Outcome': # Corrected: Compare column name
    p = dib[col].value_counts(normalize=True)   # probabilities

    # Ensure only valid probabilities are used for entropy calculation (p>0)
    # Also, handle the case where a column might have only one unique value, leading to log(0)
    entropy_val = -(p * np.log2(p.where(p > 0, 1))).sum() # Corrected: Calculate entropy inside the loop
    print(f'{col} Entropy: {entropy_val}')

NameError: name 'dib' is not defined